In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

 1. Загрузите данные из файла data-logistic.csv. Это двумерная выбор
ка, целевая переменная на которой принимает значения -1 или 1.


In [ ]:
data = np.loadtxt('data-logistic.csv', delimiter=',')
y = data[:, 0]
X = data[:, 1:]

In [ ]:
data[:10]

array([[-1.        , -0.66382654, -0.13852572],
       [ 1.        ,  1.9945955 ,  2.46802468],
       [-1.        , -1.24739492,  0.74942464],
       [ 1.        ,  2.30937425,  1.89983556],
       [ 1.        ,  0.84914331,  2.40774982],
       [ 1.        ,  1.45427095, -0.66541571],
       [ 1.        ,  2.25422743,  2.26378585],
       [-1.        , -0.06757952,  1.4691411 ],
       [-1.        , -0.86196091, -0.82485558],
       [ 1.        ,  0.69917893,  2.03248761]])

2. Убедитесь, что выше выписаны правильные формулы для градиентного спуска. Обратите внимание, что мы используем полноценный градиентный спуск, а не его стохастический вариант!

In [ ]:
# сигмоида
def sigmoid(x, w):
    return 1.0 / (1 + np.exp(-np.dot(X, w)))

# градиентный спуск
def logistic_regression(X, y, C=1e10, k=0.1, max_iter=10000, eps=1e-5):
    w = np.zeros(2)  # инициализация весов 0 0

    for i in range(max_iter):
        w_prev = w.copy()
        margins = y * (X[:,0]*w[0] + X[:,1]*w[1])
        sigmoids = 1.0 / (1 + np.exp(-margins))

        grad_w1 = np.mean(y * X[:,0] * (1 - sigmoids)) - w[0]/C
        grad_w2 = np.mean(y * X[:,1] * (1 - sigmoids)) - w[1]/C

        w[0] += k * grad_w1
        w[1] += k * grad_w2

        if np.linalg.norm(w - w_prev) <= eps: # 4 проверка сходимости
            print(f"Сходимость достигнута на итерации {i}")
            break

    return w

 3. Реализуйте градиентный спуск для обычной и L2-регуляризованной
 (с коэффициентом регуляризации 10) логистической регрессии. Используйте длину шага k=0.1. В качестве начального приближения используйте вектор (0, 0).

In [ ]:
# без регуляризации просто очень маленькая
w_no_reg = logistic_regression(X, y, C=1e10)

# L2
w_reg = logistic_regression(X, y, C=10)

Сходимость достигнута на итерации 243
Сходимость достигнута на итерации 167


In [ ]:
w_no_reg, w_reg

(array([0.28781162, 0.0919833 ]), array([0.24102099, 0.10532828]))

 4. Запустите градиентный спуск и доведите до сходимости (евклидово
 расстояние между векторами весов на соседних итерациях должно быть не больше 1e-5). Рекомендуется ограничить сверху число итераций десятью тысячами.

 5. Какое значение принимает AUC-ROC на обучении без регуляризации и при ее использовании? Эти величины будут ответом на задание. В качестве ответа приведите два числа через пробел. Обратите внимание, что на вход функции roc_auc_score нужно подавать оценки вероятностей, подсчитанные обученным алгоритмом.
 Для этого воспользуйтесь сигмоидной функцией: a(x) = 1 (1 + exp( w1x1 w2x2)).

In [ ]:
def predict_proba(X, w):
    return 1.0 / (1 + np.exp(-X[:,0]*w[0] - X[:,1]*w[1]))

auc_no_reg = roc_auc_score(y, predict_proba(X, w_no_reg))
auc_reg = roc_auc_score(y, predict_proba(X, w_reg))

round(auc_no_reg, 3), round(auc_reg, 3)

(np.float64(0.927), np.float64(0.931))

 6. Попробуйте поменять длину шага. Будет ли сходиться алгоритм, если делать более длинные шаги? Как меняется число итераций при уменьшении длины шага?

In [ ]:
for k in [0.5, 0.1, 0.01]:
    w_test = logistic_regression(X, y, C=10, k=k)

Сходимость достигнута на итерации 40
Сходимость достигнута на итерации 167
Сходимость достигнута на итерации 1023


7. Попробуйте менять начальное приближение. Влияет ли оно на что
нибудь?

Заметно, что чем ближе точка к минимуму, тем меньше итераций надо совершить, чтобы в нее прийти